# 🌊 FloodLens – Flood Level Prediction Using Deep Learning

Predicts the flood water level (Level 0 – Level 4) from a flood/car image.

**How to use:** select the **Python (floodlens)** kernel, then run the cells one by one (Shift + Enter).

> The model `floodlens_model.keras` must be in the **same folder** as this notebook.

### 1. Import Libraries

In [ ]:
import os
import io
import numpy as np
from PIL import Image, UnidentifiedImageError
from IPython.display import display
import tensorflow as tf

### 2. Load Trained Model
This loads your existing model. Nothing is retrained.

In [ ]:
# Model file name (keep it in the same folder as this notebook)
MODEL_PATH = "floodlens_model.keras"

model = None
if not os.path.exists(MODEL_PATH):
    print("❌ Model file not found!")
    print("Please place 'floodlens_model.keras' in the SAME folder as this notebook:")
    print("   ", os.getcwd())
else:
    try:
        model = tf.keras.models.load_model(MODEL_PATH)
        print("✅ FloodLens model loaded successfully.")
    except Exception as e:
        print("❌ Could not load the model file.")
        print("Details:", e)

### 3. Define Classes

In [ ]:
classes = ["Level 0", "Level 1", "Level 2", "Level 3", "Level 4"]

# The model expects 128 x 128 RGB images
IMG_SIZE = 128

### 4. Image Prediction Function

In [ ]:
# Image types we accept
SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def predict_image(img, actual_level=None):
    """Predict the flood level for a PIL image and print the result."""
    if model is None:
        print("❌ Model is not loaded. Place 'floodlens_model.keras' next to this notebook and re-run Section 2.")
        return None

    # Show the image
    print("Uploaded Image:")
    display(img.resize((300, int(300 * img.height / img.width))))

    # Prepare the image for the model
    img = img.convert("RGB")                    # make sure it has 3 colour channels
    img = img.resize((IMG_SIZE, IMG_SIZE))      # resize to 128 x 128
    arr = np.array(img, dtype="float32") / 255.0  # normalize to 0-1
    arr = np.expand_dims(arr, axis=0)           # add batch dimension -> (1, 128, 128, 3)

    # Predict
    probs = model.predict(arr, verbose=0)[0]
    best = int(np.argmax(probs))                # index of highest probability
    confidence = float(probs[best]) * 100

    # Show the result
    print("=" * 40)
    print("FloodLens Prediction")
    print("=" * 40)
    if actual_level is not None:
        print("Actual Dataset Level:", actual_level)
    print("Predicted Flood Level:", classes[best])
    print("Confidence: {:.2f}%".format(confidence))
    print("-" * 40)
    print("All class probabilities:")
    for name, p in zip(classes, probs):
        print("  {}: {:.2f}%".format(name, float(p) * 100))
    return classes[best], confidence


def predict_flood_level(image_path):
    """Predict the flood level for an image file path."""
    image_path = image_path.strip().strip('"')   # allow paths pasted with quotes

    if not os.path.exists(image_path):
        print("❌ Image file not found:", image_path)
        return None
    if not image_path.lower().endswith(SUPPORTED):
        print("❌ Unsupported image format. Please use:", ", ".join(SUPPORTED))
        return None
    try:
        img = Image.open(image_path)
        img.load()
    except (UnidentifiedImageError, OSError):
        print("❌ This file is not a valid image.")
        return None

    # If the image sits inside a folder named Level0 ... Level4, show it as the actual level
    folder = os.path.basename(os.path.dirname(image_path))
    actual = None
    if folder.lower().startswith("level") and folder[5:].isdigit():
        actual = "Level " + folder[5:]

    return predict_image(img, actual)

### 5. Test Image
Change the path below to any image on your computer, then run the cell.

Tip: in Windows Explorer, hold **Shift**, right-click the image and choose **Copy as path**.

In [ ]:
# 👇 CHANGE THIS to your image path (keep the r before the quotes)
image_path = r"C:\path\to\floodlens_dataset\val\Level3\car-1030-Level3_8.jpg"

predict_flood_level(image_path)

### 6. Simple Interactive Demo
Option A uploads an image with a button. Option B opens a normal file picker window.

**Option A – Upload button**

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    uploader = widgets.FileUpload(accept="image/*", multiple=False)
    output = widgets.Output()

    def on_upload(change):
        with output:
            clear_output()
            if not uploader.value:
                return
            # Works with both old and new ipywidgets versions
            item = uploader.value[0] if isinstance(uploader.value, (tuple, list)) else list(uploader.value.values())[0]
            content = item["content"] if isinstance(item, dict) else item.content
            try:
                img = Image.open(io.BytesIO(bytes(content)))
                img.load()
            except (UnidentifiedImageError, OSError):
                print("❌ This file is not a valid image.")
                return
            predict_image(img)

    uploader.observe(on_upload, names="value")
    print("Click the button below and choose a flood/car image:")
    display(uploader, output)
except ImportError:
    print("Upload button is not available. Please use Option B below.")

**Option B – File picker window**

Run this cell, choose an image in the window that opens (it may appear behind the browser).

In [ ]:
try:
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    chosen = filedialog.askopenfilename(
        title="Select a flood/car image",
        filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp *.webp")]
    )
    root.destroy()

    if chosen:
        predict_flood_level(chosen)
    else:
        print("No image selected.")
except Exception:
    # Last-resort simple option: type or paste the path
    predict_flood_level(input("Paste the image path here: "))

### 7. Model Results (from training)

| Item | Value |
|---|---|
| Epochs | 3 |
| Final training accuracy | 64.65% |
| Final validation accuracy | **50.66%** |
| Final validation loss | 1.1612 |

This is a small, quick-trained model, so predictions can be wrong and confidence is often low.